# Research and Policy Writing Demo

This notebook mirrors `examples/research_policy_demo.py` and runs without API keys by using a mocked OpenAI-compatible client.

The prompt is the built-in `research-answer` template filled with a policy question. The mocked draft mixes a checkable date with a fabricated citation; Chain-of-Verification flags the unsupported claim and rewrites conservatively.

In [ ]:
from unittest.mock import MagicMock

from self_correct import AntiHallucinator, templates

question = (
    "Did New York City's congestion pricing program reduce Manhattan traffic "
    "after it began in January 2025?"
)
body = templates.get_template("research-answer")
prompt, missing = templates.render(body, {"question": question})
assert prompt and not missing

def response(content, prompt_tokens=0, completion_tokens=0):
    mock = MagicMock()
    mock.choices[0].message.content = content
    mock.usage.prompt_tokens = prompt_tokens
    mock.usage.completion_tokens = completion_tokens
    return mock

draft = (
    "Yes, official counts show fewer vehicles entering the Manhattan "
    "congestion zone after the program began in January 2025.\n\n"
    "- The program started on 5 January 2025.\n"
    "- Chen et al. (2025) in Nature reported a 47% citywide traffic drop "
    "in the first month.\n\n"
    "If that 47% figure were withdrawn, an immediate citywide expansion "
    "would need to be reconsidered."
)
extracted = (
    "1. The congestion pricing program started on 5 January 2025.\n"
    "2. Chen et al. (2025) in Nature reported a 47% citywide traffic drop "
    "in the first month."
)
corrected = (
    "Yes, official counts show fewer vehicles entering the Manhattan "
    "congestion zone after the program began in January 2025.\n\n"
    "- The program started on 5 January 2025.\n\n"
    "The size of the first-month change should be taken from official MTA "
    "or city counts, not from an unsourced 47% citywide figure. If those "
    "official counts were later revised, any expansion recommendation "
    "would need to be revisited."
)

client = MagicMock()
client.chat.completions.create.side_effect = [
    response(draft, 80, 90),
    response(extracted, 70, 40),
    response('VERIFIED: True.', 30, 8),
    response(
        'VERIFIED: False. No such Nature paper exists; the 47% citywide '
        'figure is fabricated.',
        35,
        20,
    ),
    response(corrected, 90, 80),
]

agent = AntiHallucinator(
    client=client,
    strictness=1.0,
    draft_system_prompt=(
        'You are a policy analyst. Prefer attributable statistics and '
        'named sources. Do not invent citations.'
    ),
)
result = agent.generate(model='gpt-4o-mini', prompt=prompt)

print(prompt)
print()
print(result.content)
print('claims flagged:', len(result.hallucinations_caught))
print('total tokens:', result.token_usage.total_tokens)